In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.special import comb

class KMeans:
    def __init__(self, n_clusters=3, init='random', max_iter=300, random_state=42, n_init=10):
        self.n_clusters = n_clusters
        self.init = init
        self.max_iter = max_iter
        self.random_state = random_state
        self.n_init = n_init
        self.cluster_centers_ = None
        self.labels_ = None
        self.inertia_ = None

    def _euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2, axis=1))

    def _init_centroids(self, X):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        if self.init == 'random':
            indices = np.random.choice(n_samples, self.n_clusters, replace=False)
            centroids = X[indices]
        return centroids

    def _assign_clusters(self, X, centroids):
        n_samples = X.shape[0]
        labels = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            distances = self._euclidean_distance(X[i].reshape(1, -1), centroids)
            labels[i] = np.argmin(distances)
        return labels

    def _update_centroids(self, X, labels):
        n_features = X.shape[1]
        centroids = np.zeros((self.n_clusters, n_features))
        for k in range(self.n_clusters):
            cluster_samples = X[labels == k]
            if len(cluster_samples) > 0:
                centroids[k] = np.mean(cluster_samples, axis=0)
        return centroids

    def _calculate_inertia(self, X, labels, centroids):
        inertia = 0.0
        for k in range(self.n_clusters):
            cluster_samples = X[labels == k]
            if len(cluster_samples) > 0:
                inertia += np.sum((cluster_samples - centroids[k]) ** 2)
        return inertia

    def fit(self, X):
        X = np.array(X)
        n_samples, n_features = X.shape
        best_inertia = np.inf
        best_centroids = None
        best_labels = None

        for _ in range(self.n_init):
            centroids = self._init_centroids(X)
            prev_centroids = centroids.copy()

            for _ in range(self.max_iter):
                labels = self._assign_clusters(X, centroids)
                centroids = self._update_centroids(X, labels)
                if np.allclose(centroids, prev_centroids):
                    break
                prev_centroids = centroids.copy()

            inertia = self._calculate_inertia(X, labels, centroids)
            if inertia < best_inertia:
                best_inertia = inertia
                best_centroids = centroids
                best_labels = labels

        self.cluster_centers_ = best_centroids
        self.labels_ = best_labels
        self.inertia_ = best_inertia

    def predict(self, X):
        X = np.array(X)
        return self._assign_clusters(X, self.cluster_centers_)

    def fit_predict(self, X):
        self.fit(X)
        return self.labels_

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
print("===== 数据集基本信息 =====")
print(df.info())
print("\n===== 数据集描述统计 =====")
print(df.describe())
print("\n===== 各类别样本数量 =====")
print(df['target'].value_counts())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

def elbow_method(X, max_k=10):
    inertias = []
    for k in range(1, max_k+1):
        kmeans = KMeans(n_clusters=k, random_state=42)
        kmeans.fit(X)
        inertias.append(kmeans.inertia_)

    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, max_k+1), inertias, marker='o', linewidth=2)
    plt.xlabel('聚类数K')
    plt.ylabel('惯性值（簇内平方和）')
    plt.title('K-Means肘部法则：选最优K值')
    plt.grid(alpha=0.3)
    plt.axvline(x=3, color='red', linestyle='--', label='最优K=3')
    plt.legend()
    plt.show()
    return inertias

inertias = elbow_method(X_scaled)

k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans.fit(X_scaled)

cluster_labels = kmeans.labels_
centroids = kmeans.cluster_centers_
inertia = kmeans.inertia_

print(f"\n===== K-Means聚类结果（K={k}） =====")
print(f"最终质心（标准化后）：\n{centroids}")
print(f"惯性值：{inertia:.4f}")
print(f"各簇样本数量：\n{pd.Series(cluster_labels).value_counts()}")

def cluster_evaluation(true_labels, pred_labels):
    def silhouette_score(X, labels):
        n_samples = X.shape[0]
        silhouette = []
        for i in range(n_samples):
            cluster_i = X[labels == labels[i]]
            a = np.mean(np.sqrt(np.sum((cluster_i - X[i]) ** 2, axis=1))) if len(cluster_i) > 1 else 0
            b = np.inf
            for k in np.unique(labels):
                if k != labels[i]:
                    cluster_k = X[labels == k]
                    dist = np.mean(np.sqrt(np.sum((cluster_k - X[i]) ** 2, axis=1)))
                    if dist < b:
                        b = dist
            s = (b - a) / max(a, b) if max(a, b) != 0 else 0
            silhouette.append(s)
        return np.mean(silhouette)

    def adjusted_rand_score(true_labels, pred_labels):
        labels = np.unique(true_labels)
        clusters = np.unique(pred_labels)
        n = len(true_labels)
        contingency = np.zeros((len(labels), len(clusters)), dtype=int)
        for i, label in enumerate(labels):
            for j, cluster in enumerate(clusters):
                contingency[i, j] = np.sum((true_labels == label) & (pred_labels == cluster))

        sum_comb_c = sum(comb(c, 2) for c in contingency.sum(axis=0))
        sum_comb_l = sum(comb(l, 2) for l in contingency.sum(axis=1))
        sum_comb_t = sum(comb(t, 2) for t in contingency.flatten())
        comb_n = comb(n, 2)
        expected = (sum_comb_c * sum_comb_l) / comb_n
        ari = (sum_comb_t - expected) / ((sum_comb_c + sum_comb_l) / 2 - expected)
        return ari

    sil_score = silhouette_score(X_scaled, pred_labels)
    ari_score = adjusted_rand_score(true_labels, pred_labels)

    print("\n===== 聚类评估指标 =====")
    print(f"轮廓系数（Silhouette Score）：{sil_score:.4f}（越接近1越好）")
    print(f"调整兰德指数（ARI）：{ari_score:.4f}（越接近1越好）")
    return sil_score, ari_score

sil_score, ari_score = cluster_evaluation(y, cluster_labels)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
for i, target_name in enumerate(target_names):
    plt.scatter(
        X_scaled[y == i, 0], X_scaled[y == i, 1],
        label=f'真实类别：{target_name}', alpha=0.7, s=50
    )
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title('鸢尾花数据集真实类别')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
for i in range(k):
    plt.scatter(
        X_scaled[cluster_labels == i, 0], X_scaled[cluster_labels == i, 1],
        label=f'聚类簇{i+1}', alpha=0.7, s=50
    )
plt.scatter(
    centroids[:, 0], centroids[:, 1],
    c='black', marker='*', s=200, label='质心'
)
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title(f'K-Means聚类结果（K={k}）')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
for i, target_name in enumerate(target_names):
    plt.scatter(
        X_scaled[y == i, 2], X_scaled[y == i, 3],
        label=f'真实类别：{target_name}', alpha=0.7, s=50
    )
plt.xlabel(feature_names[2])
plt.ylabel(feature_names[3])
plt.title('鸢尾花数据集真实类别（花瓣特征）')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
for i in range(k):
    plt.scatter(
        X_scaled[cluster_labels == i, 2], X_scaled[cluster_labels == i, 3],
        label=f'聚类簇{i+1}', alpha=0.7, s=50
    )
plt.scatter(centroids[:, 2], centroids[:, 3], c='black', marker='*', s=200, label='质心')
plt.xlabel(feature_names[2])
plt.ylabel(feature_names[3])
plt.title(f'K-Means聚类结果（K={k}）（花瓣特征）')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()